In [ ]:
# decoupleR TF activity on the day-4 in-vitro data.
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import loompy as lp

In [ ]:
adata1 = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
adata = adata1[adata1.obs['timepoint'] == 'day4'].copy()

In [ ]:
import decoupler as dc

In [ ]:
collectri = dc.op.collectri(organism='human')
collectri

In [ ]:
dc.mt.ulm(data=adata, net=collectri)

In [ ]:
score = dc.pp.get_obsm(adata=adata, key='score_ulm')
score

In [ ]:
score

In [ ]:
df = dc.tl.rankby_group(
    adata=score,
    groupby='condition',
    reference='rest',
    method='t-test_overestim_var'
)
df = df[(df['stat'] > 0) & (df['padj'] < 0.01)]
df

In [ ]:
df.to_csv('dc_day5.csv')

In [ ]:
n_markers = 20
source_markers = (
    df
    .groupby('group')
    .head(n_markers)
    .drop_duplicates('name')
    .groupby('group')
    ['name']
    .apply(lambda x: list(x))
    .to_dict()
)
source_markers

In [ ]:
sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby='condition',
   
    standard_scale=None,
    colorbar_title='Z-scaled scores',
    cmap='RdBu_r',
    #save = 'dc_day5'
)

In [ ]:
gex_bcells = adata[adata.obs['condition'] == 'MDIX'].to_df().mean(0).to_frame().T
score_bcells = score[score.obs['condition'] == 'MDIX'].to_df().mean(0).to_frame().T


In [ ]:
dc.pl.network(
    data=gex_bcells,             # Series of gene expression
    score=score_bcells,          # Series of TF scores
    net=collectri,               # TF-target network
    sources=[
        'POLE4', 'POLE3', 'LYL1'],
    targets=20,
    size_node=5,
    figsize=(20, 20),
    s_cmap='Reds',
    save = 'dc_day4_connectivity.pdf'
)


In [ ]:
df = dc.tl.rankby_group(
    adata=score,
    groupby='condition',
    reference='rest',
    method='t-test_overestim_var'
)
df = df[df['stat'] > 0]
df

In [ ]:
df.to_csv('condition_decoupler.csv')

In [ ]:
n_markers = 10
source_markers = (
    df
    .groupby('group')
    .head(n_markers)
    .drop_duplicates('name')
    .groupby('group')
    ['name']
    .apply(lambda x: list(x))
    .to_dict()
)
source_markers

In [ ]:
sc.pl.matrixplot(
    adata=score,
    var_names=source_markers,
    groupby='condition',
   
    standard_scale='var',
    colorbar_title='Z-scaled scores',
    cmap='RdBu_r',
)

In [ ]:
gex_bcells = adata[adata.obs['condition'] == 'MDIX'].to_df().mean(0).to_frame().T
score_bcells = score[score.obs['condition'] == 'MDIX'].to_df().mean(0).to_frame().T


In [ ]:
dc.pl.network(
    data=gex_bcells,
    score=score_bcells,
    net=collectri,
    sources=['MYC', 'NPM1', 'NR3C1'],
    targets=20,
    size_node=10,
    figsize=(10, 10),
    s_cmap='Reds',
)

In [ ]:
immune_tfs = [
    "IKZF1", "BACH2", "FOXP1", "STAT4", "STAT6",
    "GFI1B", "MAFB", "LMO2", "LMO4"
]



In [ ]:
dc.pl.network(
    data=gex_bcells,             # Series of gene expression
    score=score_bcells,          # Series of TF scores
    net=collectri,               # TF-target network
    sources=immune_tfs,
    targets=20,
    size_node=10,
    figsize=(20, 20),
    s_cmap='Reds',
)
